In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/iceberg_raw_preprocessed.npz" "/content/iceberg_raw_preprocessed.npz"

In [3]:
import torch
import torch.nn as nn
import numpy as np
import random
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss

class IcebergNativeDataset(Dataset):
    def __init__(self, images, labels=None, is_train=True):
        self.images = images[:, :2, :, :]
        self.labels = labels
        self.is_train = is_train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.tensor(self.images[idx], dtype=torch.float32)
        if self.is_train:
            if random.random() > 0.5: img = torch.flip(img, [2])
            if random.random() > 0.5: img = torch.flip(img, [1])
            img = torch.rot90(img, random.randint(0, 3), [1, 2])
            shift_y, shift_x = random.randint(-4, 4), random.randint(-4, 4)
            img = torch.roll(img, shifts=(shift_y, shift_x), dims=(1, 2))

            if random.random() > 0.5:
                img = img + torch.randn_like(img) * random.uniform(0.01, 0.05)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.float32)
            return img, label
        return img

class CustomIcebergCNN(nn.Module):
    def __init__(self, dropout_rate=0.2):
        super(CustomIcebergCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(2, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate + 0.05),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate + 0.1),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout_rate + 0.1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [4]:
import numpy as np
import torch
import random
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
import torch.nn as nn
import torchvision.transforms.functional as TF
import pandas as pd

class IcebergVisionDataset(Dataset):
    def __init__(self, images, labels=None, is_train=True):
        self.images = images
        self.labels = labels
        self.is_train = is_train

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.tensor(self.images[idx], dtype=torch.float32)
        if self.is_train:
            if random.random() > 0.5: img = torch.flip(img, [2])
            if random.random() > 0.5: img = torch.flip(img, [1])
            img = torch.rot90(img, random.randint(0, 3), [1, 2])
            shift_y, shift_x = random.randint(-5, 5), random.randint(-5, 5)
            img = torch.roll(img, shifts=(shift_y, shift_x), dims=(1, 2))
            if random.random() > 0.5:
                scale_factor = random.uniform(0.9, 1.1)
                new_size = int(75 * scale_factor)
                img = TF.resize(img, [new_size, new_size], antialias=True)
                img = TF.center_crop(img, [75, 75])
            if random.random() > 0.5:
                img = img + torch.randn_like(img) * random.uniform(0.01, 0.04)

        if self.labels is not None:
            label = torch.tensor(self.labels[idx], dtype=torch.float32)
            return img, label
        return img

class IcebergResNet(nn.Module):
    def __init__(self):
        super(IcebergResNet, self).__init__()
        self.cnn = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.cnn.fc.in_features
        self.cnn.fc = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.cnn(x))

class IcebergDenseNet(nn.Module):
    def __init__(self):
        super(IcebergDenseNet, self).__init__()
        self.cnn = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        in_features = self.cnn.classifier.in_features
        self.cnn.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.cnn(x))

In [5]:
def load_train_resources(npz_path):
    print(f"Loading: {npz_path}")
    data = np.load(npz_path, allow_pickle=True)

    X_all = data['X_all'].astype(np.float32)
    angle_all = data['angle_all'].astype(np.float32).reshape(-1, 1)
    y_all = data['y_all'].astype(np.float32).reshape(-1, 1)

    if X_all.shape[1] != 3:
        X_all = X_all.transpose(0, 3, 1, 2)

    return X_all, angle_all, y_all

In [6]:
class EarlyStopping:
    def __init__(self, patience=7, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta

    def __call__(self, val_loss):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0

In [7]:
from torch.utils.data import Subset
from sklearn.model_selection import StratifiedKFold
import json
import os

X_img, X_ang, y = load_train_resources('/content/iceberg_raw_preprocessed.npz')

K_FOLDS = 5
BATCH_SIZE = 32
LR = 1e-3
WEIGHT_DECAY = 1e-3
EPOCHS = 100
PATIENCE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORKERS = min(4, os.cpu_count())

Loading: /content/iceberg_raw_preprocessed.npz


In [8]:
import pandas as pd
import numpy as np

def extract_magic_angles(train_angles, train_labels, min_samples=2, precision=4):

    df_magic = pd.DataFrame({
        'inc_angle': train_angles.flatten(),
        'is_iceberg': train_labels.flatten()
    })

    df_magic['angle_round'] = df_magic['inc_angle'].round(precision)

    stats = df_magic.groupby('angle_round').agg(
        group_count=('is_iceberg', 'count'),
        iceberg_ratio=('is_iceberg', 'mean')
    ).reset_index()

    magic_icebergs = stats[(stats['iceberg_ratio'] == 1.0) & (stats['group_count'] >= min_samples)]
    magic_ships = stats[(stats['iceberg_ratio'] == 0.0) & (stats['group_count'] >= min_samples)]

    iceberg_list = magic_icebergs['angle_round'].tolist()
    ship_list = magic_ships['angle_round'].tolist()

    print(f"Found {len(iceberg_list)} Iceberg Clusters， {len(ship_list)} Ship Clusters.")

    return iceberg_list, ship_list

AUTO_MAGIC_ICEBERGS, AUTO_MAGIC_SHIPS = extract_magic_angles(X_ang, y, min_samples=2, precision=4)

Found 132 Iceberg Clusters， 80 Ship Clusters.


In [9]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import log_loss

def train_vision_model(ModelClass, model_name, X_img, X_ang, y):
    print("\n" + "="*50)
    print(f"Start training: {model_name}")
    print("="*50)

    train_ds_master = IcebergVisionDataset(X_img, y, is_train=True)
    val_ds_master = IcebergVisionDataset(X_img, y, is_train=False)

    groups = np.round(X_ang.flatten(), 4)

    sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    oof_probs = np.zeros(len(y))
    fold_best_losses = []

    for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_img, y, groups=groups)):
        print(f"\nTraining {model_name} - FOLD {fold+1}/{K_FOLDS}")

        train_loader = DataLoader(Subset(train_ds_master, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, pin_memory=True, prefetch_factor=2)
        val_loader = DataLoader(Subset(val_ds_master, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=WORKERS, pin_memory=True)

        model = ModelClass().to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        criterion = nn.BCEWithLogitsLoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
        early_stopping = EarlyStopping(patience=PATIENCE)

        best_loss_this_fold = float('inf')
        best_val_preds = None

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0.0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(DEVICE), labels.view(-1, 1).float().to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * imgs.size(0)

            model.eval()
            val_loss = 0.0
            val_preds_epoch = []
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(DEVICE), labels.view(-1, 1).float().to(DEVICE)
                    outputs = model(imgs)
                    val_loss += criterion(outputs, labels).item() * imgs.size(0)

                    probs = torch.sigmoid(outputs).cpu().numpy().flatten()
                    val_preds_epoch.extend(probs)

            avg_train_loss = train_loss / len(train_idx)
            avg_val_loss = val_loss / len(val_idx)

            scheduler.step(avg_val_loss)

            indicator = ""
            if avg_val_loss < best_loss_this_fold:
                best_loss_this_fold = avg_val_loss
                best_val_preds = val_preds_epoch

                torch.save(model.state_dict(), f'{model_name}_fold_{fold+1}.pth')
                indicator = " [Best Save]"

            print(f"Epoch {epoch+1:02d} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}{indicator}")

            early_stopping(avg_val_loss)
            if early_stopping.early_stop:
                print(f"Fold {fold+1} Early Stopping Activated.")
                break

        fold_best_losses.append(best_loss_this_fold)
        oof_probs[val_idx] = best_val_preds

    cv_score = log_loss(y.flatten(), oof_probs)
    print(f"\n{model_name} Finish! Final OOF CV Score: {cv_score:.4f}")
    return oof_probs

In [10]:
def train_wolf_pack(X_img, X_ang, y, test_images, num_wolves=5):
    print(f"Start Training {num_wolves} Custom CNNs.")

    test_ds_native = IcebergNativeDataset(test_images, is_train=False)
    test_loader = DataLoader(test_ds_native, batch_size=32, shuffle=False)

    groups = np.round(X_ang.flatten(), 4)
    sgkf = StratifiedGroupKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

    all_wolves_oof = []
    all_wolves_test = []

    wolf_seeds = [42, 2023, 777, 999, 12345]
    wolf_dropouts = [0.15, 0.20, 0.25, 0.20, 0.15]

    for wolf_idx in range(num_wolves):
        print(f"\nTraining {wolf_idx+1} Custom CNN | Seed={wolf_seeds[wolf_idx]}, Dropout={wolf_dropouts[wolf_idx]}")

        torch.manual_seed(wolf_seeds[wolf_idx])
        random.seed(wolf_seeds[wolf_idx])
        np.random.seed(wolf_seeds[wolf_idx])

        wolf_oof_probs = np.zeros(len(y))
        wolf_test_preds = []

        for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_img, y, groups=groups)):
            train_ds_master = IcebergNativeDataset(X_img, y, is_train=True)
            val_ds_master = IcebergNativeDataset(X_img, y, is_train=False)

            train_loader = DataLoader(Subset(train_ds_master, train_idx), batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(Subset(val_ds_master, val_idx), batch_size=BATCH_SIZE, shuffle=False)

            model = CustomIcebergCNN(dropout_rate=wolf_dropouts[wolf_idx]).to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
            criterion = nn.BCEWithLogitsLoss()
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

            early_stopping = EarlyStopping(patience=PATIENCE)

            best_val_loss = float('inf')
            best_val_preds = None
            best_model_weights = None

            for epoch in range(EPOCHS):
                model.train()
                for imgs, labels in train_loader:
                    imgs, labels = imgs.to(DEVICE), labels.view(-1, 1).float().to(DEVICE)
                    optimizer.zero_grad()
                    loss = criterion(model(imgs), labels)
                    loss.backward()
                    optimizer.step()

                model.eval()
                val_loss = 0.0
                val_preds_epoch = []
                with torch.no_grad():
                    for imgs, labels in val_loader:
                        imgs, labels = imgs.to(DEVICE), labels.view(-1, 1).float().to(DEVICE)
                        outputs = model(imgs)
                        val_loss += criterion(outputs, labels).item() * imgs.size(0)
                        val_preds_epoch.extend(torch.sigmoid(outputs).cpu().numpy().flatten())

                avg_val_loss = val_loss / len(val_idx)
                scheduler.step(avg_val_loss)

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    best_val_preds = val_preds_epoch
                    best_model_weights = model.state_dict().copy()


                early_stopping(avg_val_loss)
                if early_stopping.early_stop:
                    print(f" Fold {fold+1} Activate Early Stopping (Stop in {epoch+1} epoch)")
                    break

            wolf_oof_probs[val_idx] = best_val_preds

            model.load_state_dict(best_model_weights)
            model.eval()
            fold_test_preds = []
            with torch.no_grad():
                for imgs in test_loader:
                    imgs = imgs.to(DEVICE)

                    img_v1 = imgs
                    img_v2 = torch.flip(imgs, [3])
                    img_v3 = torch.flip(imgs, [2])
                    img_v4 = torch.rot90(imgs, 1, [2, 3])


                    p1 = torch.sigmoid(model(img_v1)).cpu().numpy().flatten()
                    p2 = torch.sigmoid(model(img_v2)).cpu().numpy().flatten()
                    p3 = torch.sigmoid(model(img_v3)).cpu().numpy().flatten()
                    p4 = torch.sigmoid(model(img_v4)).cpu().numpy().flatten()


                    p_avg = (p1 + p2 + p3 + p4) / 4.0

                    fold_test_preds.extend(p_avg)

            wolf_test_preds.append(fold_test_preds)


        wolf_test_mean = np.mean(wolf_test_preds, axis=0)


        wolf_test_mean = np.mean(wolf_test_preds, axis=0)

        print(f" Number {wolf_idx+1} Custom CNN CV Score: {log_loss(y.flatten(), wolf_oof_probs):.4f}")
        all_wolves_oof.append(wolf_oof_probs)
        all_wolves_test.append(wolf_test_mean)


    pack_oof_probs = np.mean(all_wolves_oof, axis=0)
    pack_test_probs = np.mean(all_wolves_test, axis=0)
    print(f" Bagged CV Score: {log_loss(y.flatten(), pack_oof_probs):.4f}")

    return pack_oof_probs, pack_test_probs

In [11]:
def load_test_data(npz_path):
    print(f"Loading: {npz_path}")
    data_bundle = np.load(npz_path, allow_pickle=True)

    test_images = data_bundle['X_test'].astype(np.float32)
    test_angles = data_bundle['angle_test'].astype(np.float32).reshape(-1, 1)
    test_ids = data_bundle['test_id'].tolist()

    if test_images.shape[1] != 3:
        test_images = test_images.transpose(0, 3, 1, 2)

    print(f"Success! Number of Samples: {len(test_ids)}")
    return test_images, test_angles, test_ids

In [12]:
NPZ_PATH = '/content/iceberg_raw_preprocessed.npz'
test_images, test_angles, test_ids = load_test_data(NPZ_PATH)

Loading: /content/iceberg_raw_preprocessed.npz
Success! Number of Samples: 8424


In [15]:
resnet_oof_probs = train_vision_model(IcebergResNet, "resnet18", X_img, X_ang, y)

densenet_oof_probs = train_vision_model(IcebergDenseNet, "densenet121", X_img, X_ang, y)


custom_pack_oof, custom_pack_test = train_wolf_pack(X_img, X_ang, y, test_images, num_wolves=5)

from scipy.optimize import minimize

predictions = [resnet_oof_probs, densenet_oof_probs, custom_pack_oof]
model_names = ['ResNet18', 'DenseNet121', 'Custom_Wolf_Pack']
y_true = y.flatten()

def log_loss_func(weights):
    weights = weights / np.sum(weights)
    ensemble_pred = sum(w * pred for w, pred in zip(weights, predictions))
    return log_loss(y_true, ensemble_pred)

res = minimize(
    log_loss_func,
    [1/3, 1/3, 1/3],
    method='Nelder-Mead',
    bounds=[(0, 1), (0, 1), (0, 1)],
    options={'maxiter': 1000}
)
best_weights = res.x / np.sum(res.x)

for name, weight in zip(model_names, best_weights):
    print(f" {name}: {weight * 100:.2f}%")

print(f"\nFinal Weighted OOF CV Score: {res.fun:.4f}")


Start training: resnet18

Training resnet18 - FOLD 1/5
Epoch 01 | Train: 0.4535 | Val: 0.4994 [Best Save]
Epoch 02 | Train: 0.4299 | Val: 0.4282 [Best Save]
Epoch 03 | Train: 0.3809 | Val: 0.4837
Epoch 04 | Train: 0.3120 | Val: 0.5756
Epoch 05 | Train: 0.3201 | Val: 0.3044 [Best Save]
Epoch 06 | Train: 0.3016 | Val: 0.3439
Epoch 07 | Train: 0.3173 | Val: 0.4477
Epoch 08 | Train: 0.2997 | Val: 0.4726
Epoch 09 | Train: 0.2925 | Val: 0.2857 [Best Save]
Epoch 10 | Train: 0.2612 | Val: 0.3379
Epoch 11 | Train: 0.2924 | Val: 0.3574
Epoch 12 | Train: 0.2961 | Val: 0.3370
Epoch 13 | Train: 0.3592 | Val: 0.3527
Epoch 14 | Train: 0.3013 | Val: 0.2952
Epoch 15 | Train: 0.2641 | Val: 0.2680 [Best Save]
Epoch 16 | Train: 0.2405 | Val: 0.2684
Epoch 17 | Train: 0.2457 | Val: 0.2981
Epoch 18 | Train: 0.2384 | Val: 0.3852
Epoch 19 | Train: 0.2423 | Val: 0.3157
Epoch 20 | Train: 0.2424 | Val: 0.2924
Epoch 21 | Train: 0.2199 | Val: 0.2379 [Best Save]
Epoch 22 | Train: 0.2180 | Val: 0.3307
Epoch 23 | Tra

In [14]:
test_ds_vision = IcebergVisionDataset(test_images, is_train=False)
test_loader = DataLoader(test_ds_vision, batch_size=32, shuffle=False)

def get_test_predictions_tta(ModelClass, model_name):
    print(f"\n Activating {model_name} TTA Prediction")
    test_preds = []

    model = ModelClass().to(DEVICE)

    for i in range(1, K_FOLDS + 1):
        model.load_state_dict(torch.load(f'{model_name}_fold_{i}.pth', map_location=DEVICE))
        model.eval()
        fold_output = []

        with torch.no_grad():
            for imgs in test_loader:
                imgs = imgs.to(DEVICE)

                img_v1 = imgs
                img_v2 = torch.flip(imgs, [3])
                img_v3 = torch.flip(imgs, [2])
                img_v4 = torch.rot90(imgs, 1, [2, 3])


                p1 = torch.sigmoid(model(img_v1)).cpu().numpy().flatten()
                p2 = torch.sigmoid(model(img_v2)).cpu().numpy().flatten()
                p3 = torch.sigmoid(model(img_v3)).cpu().numpy().flatten()
                p4 = torch.sigmoid(model(img_v4)).cpu().numpy().flatten()


                p_avg = (p1 + p2 + p3 + p4) / 4.0

                fold_output.extend(p_avg)

        test_preds.append(fold_output)
        print(f"    Fold {i} TTA Finished")


    return np.mean(test_preds, axis=0)

resnet_test_probs = get_test_predictions_tta(IcebergResNet, "resnet18")
densenet_test_probs = get_test_predictions_tta(IcebergDenseNet, "densenet121")
final_test_probs = (
    best_weights[0] * resnet_test_probs +
    best_weights[1] * densenet_test_probs +
    best_weights[2] * custom_pack_test
)

override_iceberg_count = 0
override_ship_count = 0

for i in range(len(test_angles)):
    angle_round = np.round(test_angles[i][0], 4)

    if angle_round in AUTO_MAGIC_ICEBERGS:
        final_test_probs[i] = 0.9999
        override_iceberg_count += 1

    elif angle_round in AUTO_MAGIC_SHIPS:
        final_test_probs[i] = 0.0001
        override_ship_count += 1

submission = pd.DataFrame({'id': test_ids, 'is_iceberg': final_test_probs})
submission.to_csv('final_golden_submission.csv', index=False)
print(f" File Saved!")


 Activating resnet18 TTA Prediction
    Fold 1 TTA Finished
    Fold 2 TTA Finished
    Fold 3 TTA Finished
    Fold 4 TTA Finished
    Fold 5 TTA Finished

 Activating densenet121 TTA Prediction
    Fold 1 TTA Finished
    Fold 2 TTA Finished
    Fold 3 TTA Finished
    Fold 4 TTA Finished
    Fold 5 TTA Finished
 File Saved!
